# 00 · Environment setup + end-to-end smoke test
**Purpose:** detect the GPU/profile, verify persistence, install SGLang (own venv) + CuratorKIT/AlignTune/AuditKIT (kernel env), and run every stage on tiny data.
**Inputs:** none. **Outputs:** `requirements-*.txt` pins, `results/smoke_00.json`, smoke artifacts under `artifacts/smoke/`.
**Runtime (T4):** ~15 min on a fresh runtime (installs dominate), ~5 min after.

In [ ]:
import os, sys, json, subprocess
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
C.ensure_env()
cfg = C.load_config(); FORCE = False
GPU, P = C.gpu_profile(cfg)
print("GPU profile:", GPU, "| persistence:", C.PROJECT_ROOT, "| writable:", os.access(C.PROJECT_ROOT, os.W_OK))
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Imports + versions; pin what actually works.
import importlib.metadata as md, torch
import curatorkit, auditkit, aligntune
pipeline_pkgs = ["torch", "transformers", "trl", "peft", "datasets", "accelerate", "bitsandbytes", "litellm",
                 "curatorkit", "auditkit", "aligntune", "numpy", "pandas", "scikit-learn", "matplotlib"]
pins = {p: md.version(p) for p in pipeline_pkgs}
print(json.dumps(pins, indent=1), "| cuda:", torch.cuda.is_available())
sgl = subprocess.run([C.SGLANG_PY, "-m", "pip", "list", "--format=freeze"], capture_output=True, text=True).stdout
if not sgl:  # uv venvs have no pip
    sgl = subprocess.run(f"uv pip freeze --python {C.SGLANG_PY}", shell=True, capture_output=True, text=True).stdout
sgl_pins = [l for l in sgl.splitlines() if l.split("==")[0].lower() in {"sglang", "torch", "transformers", "flashinfer-python", "sgl-kernel", "ninja", "xgrammar"}]
(C.REPO / "requirements-sglang.txt").write_text("\n".join(sgl_pins) + "\n")
(C.REPO / "requirements-pipeline.txt").write_text("\n".join(f"{p}=={pins[p]}" for p in ["curatorkit", "auditkit", "litellm", "datasets", "numpy", "pandas", "scikit-learn", "matplotlib"]) + "\n")
(C.REPO / "requirements-train.txt").write_text("\n".join(f"{p}=={pins[p]}" for p in ["aligntune", "torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes"]) + "\n")
print("SGLang venv:", sgl_pins)

In [ ]:
# SGLang: serve the student, one chat request.
base = C.start_server(cfg["student"], "student", P["sglang_args"], mem_fraction=P["mem_fraction"])
r = C.chat(C.build_messages("Aspirin reduced 30-day mortality from 12% to 8% (p=0.01) in 400 patients.",
                            "Does aspirin reduce 30-day mortality?"), "student")
print(r)
smoke = {"sglang_chat": r["text"]}

In [ ]:
# CuratorKIT on 3 unlabelled abstracts, generator/judge = the SGLang-served student (smoke only).
from datasets import load_dataset
from curatorkit import Curator, CuratorConfig
unl = load_dataset(cfg["data"]["hf_id"], "pqa_unlabeled", split="train").select(range(3))
src = C.write_jsonl([{"instruction": r["question"], "input": C.join_contexts(r["context"]), "output": r["long_answer"]}
                     for r in unl], C.ART / "smoke" / "src3.jsonl")
fewshot = [{"question": "Is drug X associated with lower mortality?", "long_answer": "Mortality was lower with X.", "final_decision": "yes"}]
res = Curator(CuratorConfig(
    dataset=str(src), format="alpaca", llm_model="openai/student", llm_api_base=base, llm_api_key="none",
    generation_task="qa", num_questions=3, qa_prompt_template=C.qa_prompt_template(fewshot),
    hallucination_threshold=0.7, hallucination_prompt_template=C.HALLUCINATION_PROMPT, reward_threshold=0.7,
    dedup="minhash", clean=True, export_formats=["alpaca", "sharegpt"], output_dir=str(C.ART / "smoke" / "curator"),
    name="smoke")).run()
res.print_summary()
print(res.stage_counts)
for s in res.passed[:2] + res.rejected[:2]:
    print("—", getattr(s, "rejection_reason", "PASSED"), "|", s.instruction[:100], "|", s.output[-60:].replace("\n", " ⏎ "))
print(sorted(os.listdir(C.ART / "smoke" / "curator")))
smoke["curator"] = {"passed": len(res.passed), "rejected": len(res.rejected)}

In [ ]:
# AuditKIT on 5 TEST-format samples via the SGLang callable.
import auditkit as ak
lab = load_dataset(cfg["data"]["hf_id"], "pqa_labeled", split="train").select(range(5))
samples = [ak.Sample(input=C.user_message(C.join_contexts(r["context"]), r["question"]), target=r["final_decision"],
                     id=str(r["pubid"])) for r in lab]

run = ak.evaluate(samples, model=C.sglang_callable("student"), scorers=C.auditkit_scorers())
print(run.summary())
smoke["auditkit"] = run.headline
C.stop_server()

In [ ]:
# AlignTune: 5 steps of LoRA SFT on 20 rows (pqa_artificial — never TEST/DEV), backend per profile.
from aligntune.core.backend_factory import create_sft_trainer
art = load_dataset(cfg["data"]["hf_id"], "pqa_artificial", split="train").select(range(20))
rows = [{"messages": C.build_messages(C.join_contexts(r["context"]), r["question"],
                                      f"{r['long_answer']}\nAnswer: {r['final_decision']}")} for r in art]
tiny = C.write_jsonl(rows, C.ART / "smoke" / "train20.jsonl")
T = cfg["train"]
trainer = create_sft_trainer(
    model_name=cfg["student"], dataset_name=str(tiny), backend="unsloth", output_dir=str(C.ART / "smoke" / "sft"),
    num_epochs=1, max_steps=5, batch_size=2, learning_rate=T["learning_rate"], max_seq_length=T["max_seq_length"],
    use_peft=True, lora_r=T["lora_r"], lora_alpha=T["lora_alpha"], lora_dropout=T["lora_dropout"],
    lora_target_modules=T["lora_target_modules"], gradient_accumulation_steps=2, precision=P["precision"],
    seed=cfg["seed"], report_to="none", logging_steps=1, eval_strategy="no", save_strategy="no",
)
out = trainer.train()
print(out)
smoke["aligntune"] = str(out)[:500]

In [ ]:
json.dump(smoke, open(C.RESULTS / "smoke_00.json", "w"), indent=1, default=str)
print(json.dumps(smoke, indent=1, default=str))
print("Saved:", C.RESULTS / "smoke_00.json", C.ART / "smoke")